# Sprint E6 walkthrough: the hedging toolkit and its realized efficacy

In [1]:
# the repository root is importable so the package and the dashboard module can
# be imported without installing the wheel
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "efb").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
DATA = ROOT / "data"
HEDGE = DATA / "hedge"

In [2]:
# Cell 1 rule: the data hash in the results file must be the hash of the
# artifacts the criteria are read from, recomputed now, not copied
from efb import evaluate

stored = json.loads((ROOT / "sprints" / "E6" / "RESULTS.json").read_text())
assert evaluate.e6_data_hash(DATA) == stored["data_hash"], "artifact hash drift"
print("data_hash", stored["data_hash"])

data_hash 27d3ce7343ca5d1e8d2cc8bcf8d5b64e1ae9ad451a48c2f913ceae3ef90a5f61


## 1. Every criterion, its stored number and its verdict

In [3]:
# the criterion text is printed as stored, so a reworded threshold would
# show up here as a diff against sprints/E6/RESULTS.json
for name, block in stored["criteria"].items():
    print(name, block["verdict"])
    print(" ", block["criterion"])
    print(" ", json.dumps(block["stored_numbers"], sort_keys=True)[:220])

F6.1 pass
  Full FMP hedge drives every factor exposure below 1e-6 in absolute value and lifts the idio share of variance above 95%.
  {"mean_fmp_name_count": 461.89142857142855, "mean_fmp_name_count_capped": 461.89142857142855, "mean_idio_share_after_fmp": 1.0, "n_fmp_dates": 175, "worst_abs_exposure_after_fmp": 5.8323571666685226e-15, "worst_abs_expos
F6.2 pass
  ETF minimum-variance hedge removes more than 70% of the factor variance of the long-only seed book. ETFs cannot span every factor; the residual is reported.
  {"mean_factor_variance_removed_share": {"xs_v1": 0.9784935060330155, "xs_v2": 0.9784935060330155}, "mean_n_instruments": 13.028571428571428, "residual_reported": "per-factor post-hedge exposures in F6.5"}
F6.3 pass
  Realized: the hedged momentum long/short book has a beta to Mkt-RF within plus or minus 0.1 over 2018 to 2026.
  {"realized_beta_to_mkt_rf": {"xs_v1": -0.027160228642736272, "xs_v2": -0.027160228642736272}, "unhedged_realized_beta_to_mkt_rf": -0.0495262909

## 2. By hand: the beta hedge ratio

In [4]:
# h = -cov(book, SPY) / var(SPY) over the trailing 252 sessions before the
# last rebalance date, recomputed from the stored returns rather than
# copied from the metrics
from efb import eval_risk, hedge, race

wide, _counts = eval_risk.load_clean_wide(DATA)
portfolios = pd.read_parquet(DATA / "eval" / "e5_portfolios.parquet")
instrument_returns = hedge.load_etf_returns(DATA)
grid = race.race_grid(DATA)
book_returns = hedge._seed_book_returns("seed_mom_ls", wide, portfolios)
spy = instrument_returns["SPY"]
last_date = grid[-1]
h_by_hand = hedge.beta_hedge(
    book_returns.loc[book_returns.index < last_date],
    spy.loc[spy.index < last_date],
)
metrics = pd.read_parquet(HEDGE / "hedge_metrics.parquet")
stored_h = metrics.loc[
    (metrics["method"] == "beta") & (metrics["book"] == "seed_mom_ls"),
    "h_spy_beta",
].iloc[-1]
print("h by hand:", round(h_by_hand, 6))
print("h stored:", round(float(stored_h), 6))
assert abs(h_by_hand - float(stored_h)) < 1e-6

h by hand: 0.057461
h stored: 0.057461


## 3. By hand: the minimum-variance hedge normal equations

In [5]:
# h* = -(H' Sigma H)^-1 H' Sigma w, rebuilt on one grid date from the same
# pieces the engine uses, and checked against the stored positions
date = grid[-1]
names = eval_risk._window_names(wide, date)
supplied = eval_risk._xs_pieces(date, names, DATA)
betas, idio = hedge._instrument_betas(date, instrument_returns, DATA)
usable = np.isfinite(betas).all(axis=0) & np.isfinite(idio)
sigma_hh = (
    betas[:, usable].T @ supplied["factor_covariance"] @ betas[:, usable]
    + np.diag(idio[usable])
)
rows = portfolios.loc[
    (portfolios["portfolio"] == "seed_mom_ls")
    & (pd.to_datetime(portfolios["date"]) == date)
]
weights = (
    rows.set_index("ticker")["weight"].reindex(names).fillna(0.0).to_numpy()
)
exposures = supplied["design"].T @ weights
sigma_hw = betas[:, usable].T @ supplied["factor_covariance"] @ exposures
h_by_hand = hedge.min_variance_hedge(sigma_hh, sigma_hw)
positions = pd.read_parquet(HEDGE / "hedge_positions.parquet")
stored_positions = positions.loc[
    (positions["book"] == "seed_mom_ls")
    & (positions["method"] == "min_variance")
    & (positions["model"] == "xs_v1")
    & (pd.to_datetime(positions["date"]) == date)
].set_index("instrument")["weight"]
stored_h = stored_positions.reindex(np.array(hedge.INSTRUMENTS)[usable]).to_numpy()
print("normal equation residual:", float(np.abs(sigma_hh @ h_by_hand + sigma_hw).max()))
print("against stored positions:", float(np.abs(h_by_hand - stored_h).max()))
assert np.abs(sigma_hh @ h_by_hand + sigma_hw).max() < 1e-10
assert np.abs(h_by_hand - stored_h).max() < 1e-10

normal equation residual: 6.776263578034403e-21
against stored positions: 0.0


## 4. By hand: the FMP hedge is exact in model

In [6]:
# the exact hedge is -X (X'X)^-1 x, so the post-hedge exposure is zero by
# construction; the as-stored quarterly capped FMPs keep their cap drift
exact_hedge, n_names = hedge.fmp_hedge_exact(supplied["design"], exposures)
after = supplied["design"].T @ (weights + exact_hedge)
exposure_rows = pd.read_parquet(HEDGE / "e6_exposures.parquet")
capped_worst = exposure_rows["exposure_after_fmp_capped"].abs().max()
print("worst absolute exposure after the exact FMP hedge:", float(np.abs(after).max()))
print("names traded by the exact hedge:", n_names)
print("worst absolute exposure after the capped stored FMPs:", round(float(capped_worst), 4))
assert np.abs(after).max() < 1e-6

worst absolute exposure after the exact FMP hedge: 2.456368441983159e-15
names traded by the exact hedge: 496
worst absolute exposure after the capped stored FMPs: 0.7552


## 5. The D5 panel-to-column map, and the non-empty guard

In [7]:
from dashboard.tabs import d05_hedging as d5  # noqa: E402

for panel in (
    d5.headline_panel,
    d5.positions_panel,
    d5.residual_exposure_panel,
    d5.realized_panel,
    d5.decay_panel,
):
    frame = panel()
    assert not frame.empty, panel.__name__
    print(panel.__name__, frame.shape, list(frame.columns))

headline_panel (10, 8) ['book', 'method', 'model', 'factor_variance_removed_share', 'idio_share_after', 'turnover', 'cost', 'n_instruments']
positions_panel (56, 5) ['book', 'model', 'instrument', 'mean_weight', 'abs_mean_weight']
residual_exposure_panel (17, 2) ['factor', 'mean_abs_residual']
realized_panel (12, 6) ['book', 'method', 'model', 'realized_beta_to_mkt_rf', 'unhedged_realized_beta_to_mkt_rf', 'n_days']
decay_panel (6, 4) ['book', 'rebalance_frequency', 'realized_beta_to_mkt_rf', 'n_days']


## 6. Evidence for the deliverable, in citation order

In [8]:
# the numbers the study cites, in the order it cites them, all read from
# the stored artifacts rather than typed
print("worst exact FMP exposure:", exposure_rows["exposure_after_fmp"].abs().max())
print("idio share after FMP:", metrics.loc[metrics["method"] == "fmp", "idio_share_after"].mean())
print("mean FMP name count:", metrics.loc[metrics["method"] == "fmp", "name_count"].mean())
print("long-only share removed:", metrics.loc[(metrics["method"] == "min_variance") & (metrics["book"] == "seed_ew"), "factor_variance_removed_share"].mean())
print("mean instruments:", metrics.loc[metrics["method"] == "min_variance", "n_instruments"].mean())
print("momentum share removed:", metrics.loc[(metrics["method"] == "min_variance") & (metrics["book"] == "seed_mom_ls"), "factor_variance_removed_share"].mean())
print("residual per factor (top 5):")
print(exposure_rows.loc[exposure_rows["book"] == "seed_ew"].groupby("factor")["exposure_after_min_variance"].apply(lambda s: s.abs().mean()).sort_values(ascending=False).head(5))
print(pd.read_parquet(HEDGE / "e6_efficacy.parquet").to_string())
print(pd.read_parquet(HEDGE / "e6_decay.parquet").to_string())

worst exact FMP exposure: 5.8323571666685226e-15
idio share after FMP: 1.0
mean FMP name count: 461.89142857142855
long-only share removed: 0.9784935060330154
mean instruments: 13.028571428571428
momentum share removed: 0.42812909214535805
residual per factor (top 5):
factor
size         0.381570
liquidity    0.322248
sector_45    0.133682
reversal     0.115460
sector_40    0.090521
Name: exposure_after_min_variance, dtype: float64
           book        method  model  realized_beta_to_mkt_rf  unhedged_realized_beta_to_mkt_rf  n_days
0       seed_ew          beta  xs_v1                -0.055246                          0.332644   117.0
1       seed_ew          beta  xs_v2                -0.055246                          0.332644   117.0
2       seed_ew           fmp  xs_v1                 0.012490                          0.332644   565.0
3       seed_ew           fmp  xs_v2                 0.012490                          0.332644   565.0
4       seed_ew  min_variance  xs_v1        

## 7. What E7 inherits

In [9]:
# E7 builds signals on XS-v1 residuals and neutralizes them through the
# FMP machinery this sprint exercised: the exact hedge for in-model
# neutrality, the capped stored FMPs for tradability, and the realized
# efficacy record as the template for judging a hedge
print("FMP weights:", DATA / "models" / "XS-v1" / "fmp_weights.parquet")
print("exact hedge exposure floor:", float(np.abs(after).max()))
print("capped FMP drift, worst:", round(float(capped_worst), 4))
print("instrument count range:", int(metrics.loc[metrics["method"] == "min_variance", "n_instruments"].min()), int(metrics.loc[metrics["method"] == "min_variance", "n_instruments"].max()))

FMP weights: /Users/amankesarwani/PycharmProjects/equity-factor-book/data/models/XS-v1/fmp_weights.parquet
exact hedge exposure floor: 2.456368441983159e-15
capped FMP drift, worst: 0.7552
instrument count range: 12 14


## 7b. Credit port note: what changes when the instruments are bonds

In [10]:
# the same algebra ports one to one: the book's factor exposures come
# from the same design, the hedge universe swaps ETFs for rates and CDX
# contracts, and the residual is what no liquid contract spans. The
# missing-data semantics carry over unchanged: an untraded contract
# contributes zero, a held contract with no price makes the day missing
print("instruments:", hedge.INSTRUMENTS)
print("credit analogues: rates and CDX tenors in place of the sector SPDRs")
print("unspanned residual stored per factor:", bool(len(exposure_rows)))

instruments: ('SPY', 'IWM', 'QQQ', 'XLK', 'XLF', 'XLE', 'XLI', 'XLP', 'XLU', 'XLV', 'XLY', 'XLB', 'XLRE', 'XLC')
credit analogues: rates and CDX tenors in place of the sector SPDRs
unspanned residual stored per factor: True


## 8. Closing checklist

In [11]:
def numeric_leaves(node):
    out = []
    if isinstance(node, dict):
        for value in node.values():
            out.extend(numeric_leaves(value))
    elif isinstance(node, list):
        for value in node:
            out.extend(numeric_leaves(value))
    elif isinstance(node, float):
        out.append(node)
    return out


import nbformat

notebook = nbformat.read(ROOT / "notebooks" / "E6_walkthrough.ipynb", as_version=4)
source = "\n".join("".join(cell.source) for cell in notebook.cells if cell.cell_type == "code")
values = numeric_leaves(stored["criteria"])
offenders = []
for value in values:
    for text in (f"{value:.6f}", f"{value:.4f}"):
        if len(text) > 6 and text in source:
            offenders.append(text)
print("stored values typed into a cell:", offenders)
assert offenders == []
assert all(name in source for name in ("F6.1", "F6.2", "F6.3", "F6.4", "F6.5"))
print("closing checklist: clean")

stored values typed into a cell: []
closing checklist: clean
